In [1]:
from pyspark.sql import functions as F

from multitudcsd.config import get_spark_session
from multitudcsd.storage import read_delta

sesion = get_spark_session("comprobacion-route-types")

In [2]:
bronze_routes = read_delta(sesion, "bronze", "bronze_gtfs_static_routes")
print(f"[comprobaciones] {bronze_routes.count()} lineas en bronze_gtfs_static_routes")

[comprobaciones] 145 lineas en bronze_gtfs_static_routes


In [3]:
inventario = (
    bronze_routes
    .groupBy("route_type")
    .agg(
        F.count("*").alias("num_lineas"),
        F.slice(F.sort_array(F.collect_set("route_short_name")), 1, 5).alias("ejemplos"),
    )
    .orderBy(F.col("num_lineas").desc())
)

inventario.show(50, truncate=False)

+----------+----------+----------------------------+
|route_type|num_lineas|ejemplos                    |
+----------+----------+----------------------------+
|700       |76        |[100, 101, 106, 109, 110]   |
|109       |38        |[S1, S15, S2, S25, S26]     |
|900       |11        |[12, 18, M1, M10, M2]       |
|100       |10        |[FEX, RB10, RB63, RE1, RE20]|
|400       |9         |[U1, U2, U3, U4, U5]        |
|106       |1         |[RB63]                      |
+----------+----------+----------------------------+



In [5]:
import json, pathlib
from multitudcsd.ingestion.http_request import download_json

payload = download_json("https://api.viz.berlin.de/tic3/baustellen_sperrungen_tic.json")
payload["features"] = payload["features"][:3]   # recortado a 3 incidencias
pathlib.Path("tests/fixtures/viz_disruptions_sample.json").write_text(
    json.dumps(payload, ensure_ascii=False), encoding="utf-8"
)

[http] OK https://api.viz.berlin.de/tic3/baustellen_sperrungen_tic.json (745965 bytes)


FileNotFoundError: [Errno 2] No such file or directory: 'tests\\fixtures\\viz_disruptions_sample.json'